## Model Trainer

In [1]:
import os
%pwd

'd:\\Data Science\\END to END Proj\\Introvert vs Extrovert\\Introvert-Vs-Extrovert\\research'

In [2]:
os.chdir("../")

In [3]:
from dataclasses import dataclass
from pathlib import Path
@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    x_train_path: Path
    y_train_path: Path
    model_dir: Path
    xgb_model_pattern: str
    cat_model_pattern: str
    ensemble_path: Path
    # Hyper‑params & CV
    xgb_params: dict
    cat_params: dict
    n_splits: int
    n_repeats: int
    ensemble_weights: dict


In [4]:
from src.IntrovertVsExtrovert.utils.common import read_yaml, create_directories
from src.IntrovertVsExtrovert.constant import *

class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        cfg    = self.config.model_trainer
        xgb_p  = self.params.XGBoost
        cat_p  = self.params.CatBoost
        train  = self.params.training

        create_directories([cfg.root_dir, cfg.model_dir])

        return ModelTrainerConfig(
            root_dir          = Path(cfg.root_dir),
            x_train_path      = Path(cfg.x_train_path),
            y_train_path      = Path(cfg.y_train_path),
            model_dir         = Path(cfg.model_dir),
            xgb_model_pattern = cfg.xgb_model_pattern,
            cat_model_pattern = cfg.cat_model_pattern,
            ensemble_path     = Path(cfg.ensemble_path),
            xgb_params        = dict(xgb_p),
            cat_params        = dict(cat_p),
            n_splits          = train.n_splits,
            n_repeats         = train.n_repeats,
            ensemble_weights  = dict(train.ensemble_weights)
        )


In [5]:
import os, joblib, warnings, numpy as np, pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier
from pathlib import Path
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import log_loss, accuracy_score
from typing import Tuple
warnings.filterwarnings("ignore")


class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.cfg = config

    # ----------------------------------------------------------------------------------
    def _load_xy(self) -> Tuple[pd.DataFrame, pd.Series]:
        X = pd.read_parquet(self.cfg.x_train_path)
        y = pd.read_csv(self.cfg.y_train_path, header=None,encoding='utf-8').squeeze("columns")
        return X, y

    # ----------------------------------------------------------------------------------
    def _train_xgb(self, X, y) -> Tuple[np.ndarray, list]:
        oof = np.zeros(len(X))
        folds_models = []
        rkf = RepeatedStratifiedKFold(
            n_splits=self.cfg.n_splits,
            n_repeats=self.cfg.n_repeats,
            random_state=self.cfg.xgb_params["random_state"],
        )

        for fold, (tr, va) in enumerate(rkf.split(X, y)):
            dtr = xgb.DMatrix(X.iloc[tr], label=y.iloc[tr])
            dva = xgb.DMatrix(X.iloc[va], label=y.iloc[va])

            model = xgb.train(
                params=self.cfg.xgb_params,
                dtrain=dtr,
                num_boost_round=1000,
                evals=[(dva, "val")],
                early_stopping_rounds=50,
                verbose_eval=False,
            )
            oof[va] = model.predict(dva)
            mod_path = self.cfg.model_dir / self.cfg.xgb_model_pattern.format(fold=fold)
            model.save_model(mod_path)
            folds_models.append(mod_path)
            print(f"✓ XGB fold {fold} saved at {mod_path}")

        print(f"XGB log‑loss: {log_loss(y, oof):.4f}  acc: {accuracy_score(y, oof>0.5):.4f}")
        return oof, folds_models

    # ----------------------------------------------------------------------------------
    def _train_cat(self, X, y) -> Tuple[np.ndarray, list]:
        oof = np.zeros(len(X))
        folds_models = []
        rkf = RepeatedStratifiedKFold(
            n_splits=self.cfg.n_splits * 2,  # mimic notebook’s 10×2
            n_repeats=self.cfg.n_repeats,
            random_state=self.cfg.cat_params["random_seed"],
        )

        for fold, (tr, va) in enumerate(rkf.split(X, y)):
            model = CatBoostClassifier(**self.cfg.cat_params)
            model.fit(X.iloc[tr], y.iloc[tr], eval_set=(X.iloc[va], y.iloc[va]))
            oof[va] = model.predict_proba(X.iloc[va])[:, 1]
            mod_path = self.cfg.model_dir / self.cfg.cat_model_pattern.format(fold=fold)
            model.save_model(mod_path)
            folds_models.append(mod_path)
            print(f"✓ CatBoost fold {fold} saved at {mod_path}")

        print(f"Cat log‑loss: {log_loss(y, oof):.4f}  acc: {accuracy_score(y, oof>0.5):.4f}")
        return oof, folds_models

    # ----------------------------------------------------------------------------------
    def train(self):
        try:
            Path(self.cfg.model_dir).mkdir(parents=True, exist_ok=True)
            X, y = self._load_xy()

            oof_xgb, _ = self._train_xgb(X, y)
            oof_cat, _ = self._train_cat(X, y)

            # Save ensemble weights for use in Stage 5 (evaluation) & Stage 6 (prediction)
            joblib.dump(self.cfg.ensemble_weights, self.cfg.ensemble_path)
            print(f"Ensemble weights saved ➜ {self.cfg.ensemble_path}")

            # Quick blended CV metric
            w = self.cfg.ensemble_weights
            blend = w["xgb"]*oof_xgb + w["cat"]*oof_cat
            print(f"Blended CV log‑loss: {log_loss(y, blend):.4f}  acc: {accuracy_score(y, blend>0.5):.4f}")

            print("✅ Model‑training stage completed.")
        except Exception as e:
            raise RuntimeError(f"Training failed: {e}") from e


In [6]:
try:
    config = ConfigurationManager()
    modeltrainer_config  = config.get_model_trainer_config()

    trainer = ModelTrainer(modeltrainer_config)
    trainer.train()

except Exception as e:
    print(f"❌ Exception during model‑training stage: {e}")


[2025-07-12 13:26:22,339: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-07-12 13:26:22,348: INFO: common: yaml file: params.yaml loaded successfully]
[2025-07-12 13:26:22,381: INFO: common: yaml file: schema.yaml loaded successfully]
[2025-07-12 13:26:22,388: INFO: common: created directory at: artifacts]
[2025-07-12 13:26:22,392: INFO: common: created directory at: artifacts/model_training]
[2025-07-12 13:26:22,394: INFO: common: created directory at: artifacts/model_training/models]
✓ XGB fold 0 saved at artifacts\model_training\models\xgb_fold0.bin
✓ XGB fold 1 saved at artifacts\model_training\models\xgb_fold1.bin
✓ XGB fold 2 saved at artifacts\model_training\models\xgb_fold2.bin
✓ XGB fold 3 saved at artifacts\model_training\models\xgb_fold3.bin
✓ XGB fold 4 saved at artifacts\model_training\models\xgb_fold4.bin
XGB log‑loss: 0.1299  acc: 0.9684
✓ CatBoost fold 0 saved at artifacts\model_training\models\cat_fold0.cbm
✓ CatBoost fold 1 saved at artifacts\m